# Notebook 1 — Data Loading, Cleaning & Validation
**FIRE 691 | Regime-Conditioned Brinson Attribution with HMM Regime Detection**

Loads all raw data, cleans it, and saves processed parquet files for Notebooks 2 and 3.

**Run cells in order. Do not skip cells.**

---
### Files expected in `data/` folder
| File | Source |
|---|---|
| `12_Industry_Portfolios.csv` | Ken French Data Library |
| `compustat_annual_89-26.csv` | WRDS / Compustat Annual Fundamentals |
| `crsp_daily_index_90-25.csv` | WRDS / CRSP Daily Market Index |
| `crsp_delshr_90-25.csv` | WRDS / CRSP Delistings |
| `crsp_monthly_90-25.csv` | WRDS / CRSP Monthly Stock File |
| `crsp_monthly_index_90-25.csv` | WRDS / CRSP Monthly Market Index |
| `F-F_Research_Data_5_Factors_2x3.csv` | Ken French Data Library |
| `F-F_ST_Reversal_Factor.csv` | Ken French Data Library |
| `gvkey_permno_link_1986_2025.csv` | WRDS / CRSP-Compustat Link |

VIX and NBER fetched live from FRED (no API key required).


## Cell 1 — Configuration
String definitions only. No filesystem operations — drive not mounted yet.


In [ ]:
# ── CONFIGURATION — only ROOT ever needs to change ─────────────
ROOT = "YOUR_GOOGLE_DRIVE_PATH/"
# ────────────────────────────────────────────────────────────────

DATA    = ROOT + "data/"
OUTPUTS = ROOT + "outputs/"

FULL_START = "1990-01-01"
FULL_END   = "2024-12-31"
IS_START   = "1990-01-01"
IS_END     = "2003-12-31"
OOS_START  = "2004-01-01"

print(f"ROOT    : {ROOT}")
print(f"DATA    : {DATA}")
print(f"OUTPUTS : {OUTPUTS}")
print("Paths defined. Run Cell 2 to mount drive.")


## Cell 2 — Imports, Drive Mount, Output Folder
Mounts Google Drive first, then imports libraries, then creates outputs folder.
`try/except` handles already-mounted sessions.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
import requests
from io import StringIO

from google.colab import drive
try:
    drive.mount("/content/drive")
except ValueError:
    print("Drive already mounted.")

os.makedirs(OUTPUTS, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
print(f"Outputs folder ready : {OUTPUTS}")
print("Libraries loaded.")


## Cell 3 — CRSP Monthly Stock File
Source: `crsp_monthly_90-25.csv`  
Date format: `yyyy-mm-dd`  
Universe filters:  
- `ShareType == NS` (ordinary common shares, Fama & French 1993)  
- `PrimaryExch in {N, A, Q}` — NYSE, AMEX, NASDAQ (excludes R=Arca, X, B)  
- `MthPrc >= 1.00` (Hou, Xue & Zhang 2020)  
Returns already in decimal — no scaling needed.  
**No SIC column** — sourced from Compustat `sich` in Cell 10.


In [3]:
crsp_msf = pd.read_csv(
    DATA + "crsp_monthly_90-25.csv",
    usecols=["PERMNO","MthCalDt","MthRet","MthRetx",
             "MthPrc","ShrOut","PrimaryExch","ShareType"],
    low_memory=False
)

crsp_msf = crsp_msf.rename(columns={
    "PERMNO"     :"permno",
    "MthCalDt"   :"date",
    "MthRet"     :"ret",
    "MthRetx"    :"retx",
    "MthPrc"     :"prc",
    "ShrOut"     :"shrout",
    "PrimaryExch":"exchcd",
    "ShareType"  :"shrcd",
})

# Date format confirmed yyyy-mm-dd
crsp_msf["date"] = pd.to_datetime(crsp_msf["date"], format="%Y-%m-%d", errors="coerce")
crsp_msf = crsp_msf.dropna(subset=["date"])
crsp_msf = crsp_msf[
    (crsp_msf["date"] >= FULL_START) & (crsp_msf["date"] <= FULL_END)
].copy()

# Returns are already in decimal format in CIZ V2 — confirmed from raw file
# max = 5.22, mean = 0.016 in raw CSV — no scaling needed
for col in ["ret", "retx"]:
    crsp_msf[col] = pd.to_numeric(crsp_msf[col], errors="coerce")

# Winsorize at 1st and 99th percentile — standard in empirical asset pricing
# Removes data errors and reverse-split artifacts without dropping observations
# Applied before price and mktcap computation to avoid contamination
p01 = crsp_msf["ret"].quantile(0.01)
p99 = crsp_msf["ret"].quantile(0.99)
crsp_msf["ret"]  = crsp_msf["ret"].clip(lower=p01, upper=p99)
crsp_msf["retx"] = crsp_msf["retx"].clip(lower=p01, upper=p99)

print(f"ret scale confirmed : mean={crsp_msf['ret'].mean():.4f} "
      f"p01={p01:.4f}  p99={p99:.4f}  "
      f"max after winsor={crsp_msf['ret'].abs().max():.4f}")

crsp_msf["prc"]    = pd.to_numeric(crsp_msf["prc"],    errors="coerce").abs()
crsp_msf["shrout"] = pd.to_numeric(crsp_msf["shrout"], errors="coerce")
crsp_msf["mktcap"] = crsp_msf["prc"] * crsp_msf["shrout"] / 1000.0

# Universe filters
crsp_msf = crsp_msf[
    crsp_msf["exchcd"].isin(["N","A","Q"]) &
    (crsp_msf["shrcd"] == "NS") &
    (crsp_msf["prc"] >= 1.0)
].copy()

crsp_msf["month_end"] = crsp_msf["date"] + pd.offsets.MonthEnd(0)

print(f"CRSP Monthly loaded : {len(crsp_msf):,} rows | {crsp_msf['permno'].nunique():,} unique stocks")
print(f"  Date range        : {crsp_msf['date'].min().date()}  to  {crsp_msf['date'].max().date()}")


ret scale confirmed : mean=0.0058 p01=-0.4107  p99=0.5600  max after winsor=0.5600
CRSP Monthly loaded : 2,444,538 rows | 23,466 unique stocks
  Date range        : 1990-01-31  to  2024-12-31


## Cell 4 — CRSP Delistings
Source: `crsp_delshr_90-25.csv`  
Date format: `yyyy-mm-dd`  
Shumway (1997) imputation by `DelReasonType`: −0.55 liquidation | −0.30 performance | 0.00 transfer


In [4]:
crsp_del = pd.read_csv(
    DATA + "crsp_delshr_90-25.csv",
    usecols=["PERMNO","DelistingDt","DelRet","DelReasonType"],
    low_memory=False
)
crsp_del = crsp_del.rename(columns={
    "PERMNO"       :"permno",
    "DelistingDt"  :"dlstdt",
    "DelRet"       :"dlret",
    "DelReasonType":"dlstcd",
})

# Date format confirmed yyyy-mm-dd
crsp_del["dlstdt"] = pd.to_datetime(crsp_del["dlstdt"], format="%Y-%m-%d", errors="coerce")
crsp_del = crsp_del.dropna(subset=["dlstdt"])
crsp_del["dlret"] = pd.to_numeric(crsp_del["dlret"], errors="coerce")

IMPUTE_LIQUIDATION = {"BKPY","LP"}
IMPUTE_PERFORMANCE = {
    "INSC","INSF","EQRQ","FARG","FING","DELQ","CORQ",
    "VIO","SERQ","DEEX","SHLD","MTMK","DERE","OFFRE","UNAV"
}
IMPUTE_TRANSFER = {"MVOT","MVCHI","MVPAC","MVTO","MVMF","FDCV","PUBI"}

missing = crsp_del["dlret"].isna()
crsp_del.loc[missing & crsp_del["dlstcd"].isin(IMPUTE_LIQUIDATION), "dlret"] = -0.55
crsp_del.loc[missing & crsp_del["dlstcd"].isin(IMPUTE_PERFORMANCE), "dlret"] = -0.30
crsp_del.loc[missing & crsp_del["dlstcd"].isin(IMPUTE_TRANSFER),    "dlret"] =  0.00
crsp_del.loc[crsp_del["dlret"].isna(), "dlret"] = -0.30

print(f"Delistings loaded   : {len(crsp_del):,} rows | missing imputed: {missing.sum():,}")


Delistings loaded   : 21,363 rows | missing imputed: 1,244


## Cell 5 — CRSP Monthly Market Index
Source: `crsp_monthly_index_90-25.csv` | Date format: `yyyy-mm-dd`


In [5]:
crsp_mi = pd.read_csv(DATA + "crsp_monthly_index_90-25.csv", low_memory=False)
crsp_mi = crsp_mi.rename(columns={"MthCalDt":"date"})

# Date format confirmed yyyy-mm-dd
crsp_mi["date"] = pd.to_datetime(crsp_mi["date"], format="%Y-%m-%d", errors="coerce")
crsp_mi = crsp_mi.dropna(subset=["date"])
crsp_mi = crsp_mi[
    (crsp_mi["date"] >= FULL_START) & (crsp_mi["date"] <= FULL_END)
].copy()
for col in ["vwretd","ewretd"]:
    if col in crsp_mi.columns:
        crsp_mi[col] = pd.to_numeric(crsp_mi[col], errors="coerce")
        if crsp_mi[col].abs().max() > 1.5:
            crsp_mi[col] /= 100.0
crsp_mi["month_end"] = crsp_mi["date"] + pd.offsets.MonthEnd(0)
print(f"CRSP Monthly Index  : {len(crsp_mi):,} rows")


CRSP Monthly Index  : 420 rows


## Cell 6 — CRSP Daily Market Index
Source: `crsp_daily_index_90-25.csv` | Date format: `yyyy-mm-dd`  
Primary HMM fitting base (~3,500 IS daily obs vs 168 monthly).


In [6]:
crsp_di = pd.read_csv(
    DATA + "crsp_daily_index_90-25.csv",
    usecols=["DlyCalDt","vwretd","ewretd"],
    low_memory=False
)
crsp_di = crsp_di.rename(columns={"DlyCalDt":"date"})

# Date format confirmed yyyy-mm-dd
crsp_di["date"] = pd.to_datetime(crsp_di["date"], format="%Y-%m-%d", errors="coerce")
crsp_di = crsp_di.dropna(subset=["date"])
crsp_di = crsp_di[
    (crsp_di["date"] >= FULL_START) & (crsp_di["date"] <= FULL_END)
].copy()
for col in ["vwretd","ewretd"]:
    crsp_di[col] = pd.to_numeric(crsp_di[col], errors="coerce")
    if crsp_di[col].abs().max() > 1.5:
        crsp_di[col] /= 100.0
crsp_di_s = crsp_di.sort_values("date").reset_index(drop=True)
print(f"CRSP Daily Index    : {len(crsp_di_s):,} trading days")


CRSP Daily Index    : 8,817 trading days


## Cell 7 — Compustat Annual Fundamentals
Source: `compustat_annual_89-26.csv` | Date format: `yyyy-mm-dd`  
GP/Assets = (revt − cogs) / at, Novy-Marx (2013). December fiscal year ends only.


In [7]:
comp_raw = pd.read_csv(
    DATA + "compustat_annual_89-26.csv",
    usecols=["gvkey","datadate","fyear","at","cogs","revt","sich"],
    low_memory=False
)

comp_raw["datadate"] = pd.to_datetime(comp_raw["datadate"], format="%Y-%m-%d", errors="coerce")
comp_raw = comp_raw.dropna(subset=["datadate"])
comp_raw = comp_raw[
    (comp_raw["datadate"] >= "1989-01-01") & (comp_raw["datadate"] <= FULL_END)
].copy()

for col in ["revt","cogs","at"]:
    comp_raw[col] = pd.to_numeric(comp_raw[col], errors="coerce")

comp_raw["gp"]    = comp_raw["revt"] - comp_raw["cogs"]
comp_raw["at"]    = comp_raw["at"].replace(0, np.nan)
comp_raw["gp_at"] = np.where(
    (comp_raw["gp"] > 0) & (comp_raw["at"] > 0),
    comp_raw["gp"] / comp_raw["at"], np.nan
)

# ── ALL fiscal year ends — NOT December only ─────────────────────
# Novy-Marx (2013) and Fama-French use all fiscal year ends with a
# 6-month minimum filing lag. For the July-year-t rebalance we use
# data from fiscal years ending in calendar year t-1 (rebal_year =
# fyear + 1), giving a minimum 6-month lag for December FY ends and
# a longer lag for earlier FY ends. This includes Apple (Sep FY),
# Microsoft (Jun FY), and all other non-December fiscal year firms.
# Excluding December-only was a data construction error that
# systematically removed the largest technology and consumer firms.

comp_raw["fiscal_month"] = comp_raw["datadate"].dt.month
comp_raw["fiscal_year"]  = comp_raw["fyear"].astype("Int64")

# Keep one observation per gvkey per fiscal year (latest datadate)
comp_raw = (
    comp_raw.sort_values("datadate")
    .groupby(["gvkey","fiscal_year"]).last().reset_index()
)

comp_raw["sich"]  = pd.to_numeric(comp_raw["sich"],  errors="coerce").astype("Int64")
comp_raw["gvkey"] = comp_raw["gvkey"].astype(str).str.zfill(6)

print(f"Compustat loaded    : {len(comp_raw):,} firm-years")
print(f"  Date range        : {comp_raw['datadate'].min().date()}  to  {comp_raw['datadate'].max().date()}")
print(f"  GP/Assets coverage: {comp_raw['gp_at'].notna().mean():.1%}")
print(f"  sich coverage     : {comp_raw['sich'].notna().mean():.1%}")
print(f"\nFiscal month breakdown (all months now included):")
print(comp_raw["fiscal_month"].value_counts().sort_index().to_string())

Compustat loaded    : 413,542 firm-years
  Date range        : 1989-01-31  to  2024-12-31
  GP/Assets coverage: 67.5%
  sich coverage     : 70.5%

Fiscal month breakdown (all months now included):
fiscal_month
1       9967
2       4505
3      20340
4       6545
5       5798
6      24303
7       5856
8       7148
9      21305
10     10826
11      5227
12    291722


## Cell 8 — CRSP-Compustat Link Table
Source: `gvkey_permno_link_1986_2025.csv`  
Date format: `m/d/yyyy` (e.g. 1/31/1987) for both `datadate` and `fy_end`


In [8]:
link = pd.read_csv(DATA + "gvkey_permno_link_1986_2025.csv", low_memory=False)

# Date format confirmed m/d/yyyy — %m/%d/%Y handles both 1/31/1987 and 01/31/1987
for col in ["datadate","fy_end"]:
    if col in link.columns:
        link[col] = pd.to_datetime(link[col], format="%m/%d/%Y", errors="coerce")

link["fyear"]  = pd.to_numeric(link["fyear"],  errors="coerce").astype("Int64")
link["permno"] = pd.to_numeric(link["permno"], errors="coerce").astype("Int64")
link["gvkey"]  = link["gvkey"].astype(str).str.zfill(6)
link = link.dropna(subset=["permno","gvkey"])
print(f"Link table loaded   : {len(link):,} rows | {link['permno'].nunique():,} unique permnos")


Link table loaded   : 325,196 rows | 31,802 unique permnos


## Cell 9 — FF12 Industry Returns
Source: `12_Industry_Portfolios.csv` (Ken French, no skip rows needed)  
First column: date (no label), format yyyymm. Returns stored as percent in source.


In [9]:
ff12_raw = pd.read_csv(DATA + "12_Industry_Portfolios.csv", header=0)
ff12_raw = ff12_raw.rename(columns={ff12_raw.columns[0]: "date"})
ff12_raw["date"] = pd.to_datetime(ff12_raw["date"].astype(str), format="%Y%m")
ff12_raw["date"] = ff12_raw["date"] + pd.offsets.MonthEnd(0)
INDUSTRIES = [c for c in ff12_raw.columns if c != "date"]
for col in INDUSTRIES:
    ff12_raw[col] = pd.to_numeric(ff12_raw[col], errors="coerce")
    ff12_raw[col] = ff12_raw[col].replace([-99.99,-999.0,-9.99], np.nan)
    ff12_raw[col] /= 100.0
ff12_ret = ff12_raw[
    (ff12_raw["date"] >= FULL_START) & (ff12_raw["date"] <= FULL_END)
].copy().reset_index(drop=True)
print(f"FF12 loaded         : {len(ff12_ret):,} months | {INDUSTRIES}")


FF12 loaded         : 420 months | ['NoDur', 'Durbl', 'Manuf', 'Enrgy', 'Chems', 'BusEq', 'Telcm', 'Utils', 'Shops', 'Hlth', 'Money', 'Other']


## Cell 10 — SIC Code Assignment and FF12 Classification
SIC sourced from Compustat `sich` via link table (CIZ V2 monthly file has no SIC column).  
FF12 ranges follow `Siccodes12.txt` verbatim.  
Chems gap at 2830-2839 → pharma falls to Hlth | BusEq skips 3693 → falls to Hlth | Shops = full 5000-5999


In [10]:
# Step 1: permno -> SIC via Compustat + link
sic_lookup = (
    comp_raw[comp_raw["sich"].notna()]
    .sort_values("datadate")
    .groupby("gvkey")["sich"].last().reset_index()
)
link_sic = link[["gvkey","permno"]].drop_duplicates(subset="permno", keep="last")
sic_linked = (
    sic_lookup.merge(link_sic, on="gvkey", how="inner")
    .dropna(subset=["permno"])
)
sic_linked["permno"] = sic_linked["permno"].astype("Int64")
crsp_msf = crsp_msf.merge(sic_linked[["permno","sich"]], on="permno", how="left")
crsp_msf["siccd"] = crsp_msf["sich"]

# Step 2: FF12 classification function (Siccodes12.txt verbatim)
def sic_to_ff12(sic):
    if pd.isna(sic): return "Other"
    s = int(sic)
    if (100<=s<=999 or 2000<=s<=2399 or 2700<=s<=2749
            or 2770<=s<=2799 or 3100<=s<=3199 or 3940<=s<=3989): return "NoDur"
    if (2500<=s<=2519 or 2590<=s<=2599 or 3630<=s<=3659
            or 3710<=s<=3711 or s==3714 or s==3716
            or 3750<=s<=3751 or s==3792
            or 3900<=s<=3939 or 3990<=s<=3999): return "Durbl"
    if (2520<=s<=2589 or 2600<=s<=2699 or 2750<=s<=2769
            or 3000<=s<=3099 or 3200<=s<=3569 or 3580<=s<=3629
            or 3700<=s<=3709 or 3712<=s<=3713 or s==3715
            or 3717<=s<=3749 or 3752<=s<=3791 or 3793<=s<=3799
            or 3830<=s<=3839 or 3860<=s<=3899): return "Manuf"
    if 1200<=s<=1399 or 2900<=s<=2999: return "Enrgy"
    if 2800<=s<=2829 or 2840<=s<=2899: return "Chems"
    if (3570<=s<=3579 or 3660<=s<=3692 or 3694<=s<=3699
            or 3810<=s<=3829 or 7370<=s<=7379): return "BusEq"
    if 4800<=s<=4899: return "Telcm"
    if 4900<=s<=4949: return "Utils"
    if 5000<=s<=5999 or 7200<=s<=7299 or 7600<=s<=7699: return "Shops"
    if 2830<=s<=2839 or s==3693 or 3840<=s<=3859 or 8000<=s<=8099: return "Hlth"
    if 6000<=s<=6999: return "Money"
    return "Other"

# Step 3: Apply
crsp_msf["ff12"] = crsp_msf["siccd"].apply(sic_to_ff12)
print(f"FF12 classification : {(crsp_msf['ff12']!='Other').mean():.1%} of stock-months mapped")


FF12 classification : 75.9% of stock-months mapped


## Cell 11 — Fama-French 5 Factors
Source: `F-F_Research_Data_5_Factors_2x3.csv` | First column: date (no label), format yyyymm

In [11]:
ff5_raw = pd.read_csv(DATA + "F-F_Research_Data_5_Factors_2x3.csv", header=0)
ff5_raw = ff5_raw.rename(columns={ff5_raw.columns[0]: "date"})
ff5_raw["date"] = pd.to_datetime(ff5_raw["date"].astype(str), format="%Y%m")
ff5_raw["date"] = ff5_raw["date"] + pd.offsets.MonthEnd(0)
for col in [c for c in ff5_raw.columns if c != "date"]:
    ff5_raw[col] = pd.to_numeric(ff5_raw[col], errors="coerce")
    ff5_raw[col] = ff5_raw[col].replace([-99.99,-999.0], np.nan)
    ff5_raw[col] /= 100.0
ff5_raw = ff5_raw[
    (ff5_raw["date"] >= FULL_START) & (ff5_raw["date"] <= FULL_END)
].copy().reset_index(drop=True)
print(f"FF5 loaded          : {len(ff5_raw):,} months | columns: {list(ff5_raw.columns)}")


FF5 loaded          : 420 months | columns: ['date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']


## Cell 12 — Short-Term Reversal Factor
Source: `F-F_ST_Reversal_Factor.csv` | First column: date (no label), format yyyymm

In [12]:
strev_raw = pd.read_csv(DATA + "F-F_ST_Reversal_Factor.csv", header=0)
strev_raw = strev_raw.rename(columns={strev_raw.columns[0]: "date"})
strev_raw["date"]   = pd.to_datetime(strev_raw["date"].astype(str), format="%Y%m")
strev_raw["date"]   = strev_raw["date"] + pd.offsets.MonthEnd(0)
strev_raw["ST_Rev"] = pd.to_numeric(strev_raw["ST_Rev"], errors="coerce")
strev_raw["ST_Rev"] = strev_raw["ST_Rev"].replace([-99.99,-999.0], np.nan)
strev_raw["ST_Rev"] /= 100.0
strev_raw = strev_raw[
    (strev_raw["date"] >= FULL_START) & (strev_raw["date"] <= FULL_END)
].copy().reset_index(drop=True)
print(f"STREV loaded        : {len(strev_raw):,} months")


STREV loaded        : 420 months


## Cell 13 — VIX from FRED (Daily & Monthly)
Public FRED URL — no API key required. Falls back to cached parquet if unreachable.


In [13]:
FRED_VIX = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS"
try:
    resp = requests.get(FRED_VIX, timeout=30)
    resp.raise_for_status()
    vix_raw = pd.read_csv(StringIO(resp.text))
    vix_raw.columns = ["date","VIXCLS"]
    vix_raw["date"]   = pd.to_datetime(vix_raw["date"], errors="coerce")
    vix_raw["VIXCLS"] = pd.to_numeric(vix_raw["VIXCLS"], errors="coerce")
    vix_raw = vix_raw.dropna()
    print(f"VIX fetched from FRED: {len(vix_raw):,} daily obs")
except Exception as e:
    print(f"FRED fetch failed ({e}) — loading from cache.")
    vix_raw = pd.read_parquet(OUTPUTS + "vix_daily.parquet")
vix_raw   = vix_raw[
    (vix_raw["date"] >= FULL_START) & (vix_raw["date"] <= FULL_END)
].copy()
vix_daily = vix_raw.sort_values("date").reset_index(drop=True)
vix_monthly = (
    vix_raw
    .assign(month_end=lambda x: x["date"] + pd.offsets.MonthEnd(0))
    .groupby("month_end")["VIXCLS"].last().reset_index()
    .rename(columns={"month_end":"date"})
)
print(f"VIX daily: {len(vix_daily):,} obs | monthly: {len(vix_monthly):,} obs")


VIX fetched from FRED: 9,176 daily obs
VIX daily: 8,834 obs | monthly: 420 obs


## Cell 14 — Realized Volatility (Daily & Monthly)
21-day rolling std of daily vwretd × sqrt(252). Daily series is the second HMM input.


In [14]:
crsp_di_s["realvol_21d"] = (
    crsp_di_s["vwretd"]
    .rolling(window=21, min_periods=15)
    .std() * np.sqrt(252)
)
realvol_daily = crsp_di_s[["date","vwretd","realvol_21d"]].dropna(
    subset=["realvol_21d"]
).reset_index(drop=True)
realvol_daily["month_end"] = realvol_daily["date"] + pd.offsets.MonthEnd(0)
realvol_monthly = (
    realvol_daily
    .groupby("month_end")["vwretd"]
    .agg(lambda x: x.std() * np.sqrt(252 * len(x) / 21))
    .reset_index()
    .rename(columns={"month_end":"date","vwretd":"realvol"})
)
print(f"Realized vol daily  : {len(realvol_daily):,} obs | monthly: {len(realvol_monthly):,} obs")


Realized vol daily  : 8,803 obs | monthly: 420 obs


## Cell 15 — NBER Recession Dates from FRED
Public FRED URL — no API key required.

In [15]:
FRED_NBER = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=USREC"
try:
    resp = requests.get(FRED_NBER, timeout=30)
    resp.raise_for_status()
    nber_raw = pd.read_csv(StringIO(resp.text))
    nber_raw.columns = ["date","recession"]
    nber_raw["date"]      = pd.to_datetime(nber_raw["date"], errors="coerce")
    nber_raw["recession"] = pd.to_numeric(nber_raw["recession"], errors="coerce")
    nber_raw = nber_raw.dropna()
    print(f"NBER fetched from FRED: {len(nber_raw):,} monthly obs")
except Exception as e:
    print(f"FRED fetch failed ({e}) — loading from cache.")
    nber_raw = pd.read_parquet(OUTPUTS + "nber.parquet")
nber_raw = nber_raw[
    (nber_raw["date"] >= FULL_START) & (nber_raw["date"] <= FULL_END)
].copy()
print(f"NBER loaded         : {nber_raw['recession'].sum():.0f} recession months ({nber_raw['recession'].mean():.1%})")


NBER fetched from FRED: 2,057 monthly obs
NBER loaded         : 36 recession months (8.6%)


## Cell 16 — Merge GP/Assets onto CRSP
FF (1992) July rebalancing: December year t-1 → available July year t.  
Link table maps gvkey + fyear → permno directly. Vectorised expansion.


In [16]:
link_fyear = (
    link[["gvkey","fyear","permno"]]
    .drop_duplicates(subset=["gvkey","fyear"], keep="last")
    .copy()
)
comp_gp = (
    comp_raw[["gvkey","fiscal_year","gp_at"]]
    .merge(link_fyear, left_on=["gvkey","fiscal_year"],
           right_on=["gvkey","fyear"], how="left")
    .dropna(subset=["permno"])
    .copy()
)
comp_gp["permno"] = comp_gp["permno"].astype("Int64")

# Vectorised expansion: Dec year t -> July t+1 through June t+2
month_slots = [(0,m) for m in range(7,13)] + [(1,m) for m in range(1,7)]
frames = []
for yr_off, mo in month_slots:
    tmp = comp_gp[["permno","fiscal_year","gp_at"]].copy()
    yr  = (tmp["fiscal_year"] + 1 + yr_off).astype(int)
    tmp["month_end"] = pd.to_datetime(
        yr.astype(str) + f"-{mo:02d}-01"
    ) + pd.offsets.MonthEnd(0)
    frames.append(tmp[["permno","month_end","gp_at"]])
gp_monthly = pd.concat(frames, ignore_index=True)
gp_monthly["permno"] = gp_monthly["permno"].astype("Int64")

crsp_msf = crsp_msf.merge(
    gp_monthly[["permno","month_end","gp_at"]],
    on=["permno","month_end"], how="left"
)
print(f"GP/Assets merged    : {crsp_msf['gp_at'].notna().mean():.1%} stock-month coverage")


GP/Assets merged    : 75.6% stock-month coverage


## Cell 17 — Consolidated Summary Output
All diagnostics in one place. Run after all loading cells complete.

In [17]:
SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  NOTEBOOK 1 — FULL DATASET SUMMARY")
print(SEP)

print(f"\n{'CRSP MONTHLY STOCK FILE':^70}")
print(SEP2)
print(f"  Rows            : {len(crsp_msf):,}")
print(f"  Unique stocks   : {crsp_msf['permno'].nunique():,}")
print(f"  Date range      : {crsp_msf['date'].min().date()}  to  {crsp_msf['date'].max().date()}")
print(f"  Missing ret     : {crsp_msf['ret'].isna().mean():.2%}")
print(f"  GP/At coverage  : {crsp_msf['gp_at'].notna().mean():.2%}")
print(f"  SIC coverage    : {crsp_msf['siccd'].notna().mean():.2%}")
print(f"  Exchange split  : {dict(crsp_msf['exchcd'].value_counts())}")

print(f"\n{'CRSP DELISTINGS':^70}")
print(SEP2)
print(f"  Total records   : {len(crsp_del):,}")
print(f"  Reason breakdown:\n{crsp_del['dlstcd'].value_counts().to_string()}")

print(f"\n{'FF12 INDUSTRY RETURNS':^70}")
print(SEP2)
print(f"  Months     : {len(ff12_ret):,}")
print(f"  Date range : {ff12_ret['date'].min().date()}  to  {ff12_ret['date'].max().date()}")
print(f"  {'Industry':<10} {'Mean':>8} {'Std':>8} {'Missing':>8}")
for ind in INDUSTRIES:
    print(f"  {ind:<10} {ff12_ret[ind].mean():>8.4f} {ff12_ret[ind].std():>8.4f} {ff12_ret[ind].isna().sum():>8}")

print(f"\n{'HMM INPUT SERIES (DAILY)':^70}")
print(SEP2)
is_days  = crsp_di_s[crsp_di_s["date"] <= IS_END].shape[0]
oos_days = crsp_di_s[crsp_di_s["date"] >= OOS_START].shape[0]
print(f"  CRSP daily index   : {len(crsp_di_s):,} obs | IS: {is_days:,} | OOS: {oos_days:,}")
print(f"  Realized vol daily : {len(realvol_daily):,} obs")
print(f"  VIX daily          : {len(vix_daily):,} obs | mean={vix_daily['VIXCLS'].mean():.2f} max={vix_daily['VIXCLS'].max():.2f}")

print(f"\n{'MACRO SERIES':^70}")
print(SEP2)
peak_rv = realvol_monthly.loc[realvol_monthly['realvol'].idxmax(),'date'].date()
print(f"  Realized vol monthly: {len(realvol_monthly):,} obs | mean={realvol_monthly['realvol'].mean():.4f} max={realvol_monthly['realvol'].max():.4f} ({peak_rv})")
print(f"  NBER recessions     : {nber_raw['recession'].sum():.0f} months ({nber_raw['recession'].mean():.1%} of sample)")

print(f"\n{'FF12 STOCK COVERAGE (JULY SNAPSHOTS)':^70}")
print(SEP2)
july_crsp         = crsp_msf[crsp_msf["date"].dt.month == 7].copy()
july_crsp["year"] = july_crsp["date"].dt.year
coverage = (
    july_crsp.groupby(["year","ff12"])["permno"].nunique()
    .unstack("ff12").fillna(0).astype(int)
)
print(f"  Min stocks in any industry-year : {coverage.values.min()}")
print(f"  Mean stocks per industry-year   : {coverage.values.mean():.0f}")
print(f"  Industry-years < 10 stocks      : {(coverage.values<10).sum()} / {coverage.size}")
print(f"  Industry-years <  5 stocks      : {(coverage.values<5).sum()}  / {coverage.size}")
print("\nLast 5 years of stock counts per FF12 industry:")
print(coverage.tail(5).to_string())

print(f"\n{'FF12 CLASSIFICATION — UNIQUE PERMNOS PER INDUSTRY':^70}")
print(SEP2)
ff12_counts = crsp_msf.groupby("ff12")["permno"].nunique().sort_values(ascending=False)
for ind, cnt in ff12_counts.items():
    print(f"  {ind:<10} : {cnt:,}")

print(f"\n{SEP}")
print("  ALL DATASETS LOADED SUCCESSFULLY — READY TO SAVE")
print(SEP)


  NOTEBOOK 1 — FULL DATASET SUMMARY

                       CRSP MONTHLY STOCK FILE                        
----------------------------------------------------------------------
  Rows            : 2,444,656
  Unique stocks   : 23,466
  Date range      : 1990-01-31  to  2024-12-31
  Missing ret     : 0.04%
  GP/At coverage  : 75.57%
  SIC coverage    : 87.36%
  Exchange split  : {'Q': np.int64(1404041), 'N': np.int64(829729), 'A': np.int64(210886)}

                           CRSP DELISTINGS                            
----------------------------------------------------------------------
  Total records   : 21,363
  Reason breakdown:
dlstcd
UNAV     13929
FING      1414
LP        1195
INSC       935
BKPY       744
DELQ       659
CORQ       655
INSF       610
MVOT       327
PUBI       198
MTMK       182
EQRQ       157
SHLD       118
DERE        93
FDCV        46
FARG        39
VIO         19
OFFRE       12
MVMF        12
SERQ         8
DEEX         6
MVPAC        2
MVTO         2
MVCH

## Cell 18 — Save All Processed Files
Saves 14 parquet files to `outputs/`.  
**Run this cell last.** Notebooks 2 and 3 read from these files.


In [18]:
save_map = {
    "crsp_msf"          : crsp_msf,
    "crsp_monthly_index": crsp_mi,
    "crsp_daily_index"  : crsp_di_s,
    "crsp_delshr"       : crsp_del,
    "compustat"         : comp_raw,
    "link"              : link,
    "ff12_returns"      : ff12_ret,
    "ff5_factors"       : ff5_raw,
    "strev_factor"      : strev_raw,
    "vix_daily"         : vix_daily,
    "vix_monthly"       : vix_monthly,
    "realvol_daily"     : realvol_daily,
    "realvol_monthly"   : realvol_monthly,
    "nber"              : nber_raw,
}

print("Saving processed files to outputs/ ...")
for name, df in save_map.items():
    path = OUTPUTS + f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"  {name}.parquet  ({len(df):,} rows)  saved")

print(f"\nAll {len(save_map)} files saved to:")
print(f"  {OUTPUTS}")
print("\nNotebooks 2 and 3 are ready to run.")


Saving processed files to outputs/ ...
  crsp_msf.parquet  (2,444,656 rows)  saved
  crsp_monthly_index.parquet  (420 rows)  saved
  crsp_daily_index.parquet  (8,817 rows)  saved
  crsp_delshr.parquet  (21,363 rows)  saved
  compustat.parquet  (413,542 rows)  saved
  link.parquet  (325,196 rows)  saved
  ff12_returns.parquet  (420 rows)  saved
  ff5_factors.parquet  (420 rows)  saved
  strev_factor.parquet  (420 rows)  saved
  vix_daily.parquet  (8,834 rows)  saved
  vix_monthly.parquet  (420 rows)  saved
  realvol_daily.parquet  (8,803 rows)  saved
  realvol_monthly.parquet  (420 rows)  saved
  nber.parquet  (420 rows)  saved

All 14 files saved to:
  /content/drive/MyDrive/Colab Notebooks/FIRE-691-901_Adv_Financial_Analytics/Brinson-Carino Attribution - HMM regime detection/outputs/

Notebooks 2 and 3 are ready to run.
